In [2]:
from deepagents import create_deep_agent
from dotenv import load_dotenv
load_dotenv()
import os

/Users/nitinaggarwal/Documents/learning/langgraph_deep_agents/.venv/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
nvidia_api_key = os.getenv("NVIDIA_API_KEY")
print(nvidia_api_key[:5])

nvapi


| Parameter                                                                         | What it does                                                                |
| --------------------------------------------------------------------------------- | --------------------------------------------------------------------------- |
| [`model=`](#model)                                                                | Which model to use                                                          |
| [`system_prompt=`](#system-prompt)                                                | Custom instructions for the agent                                           |
| [`tools=`](#tools)                                                                | Domain tools the agent can call                                             |
| [`memory=`](#memory)                                                              | AGENTS.md files loaded at startup                                           |
| [`skills=`](#skills)                                                              | Skills directory for on-demand knowledge                                    |
| [`backend=`](#backends)                                                           | Filesystem backend (StateBackend by default)                                |
| [`permissions=`](/oss/python/deepagents/permissions)                              | Path-level access control for the filesystem                                |
| [`subagents=`](#subagents)                                                        | Custom subagents for delegated tasks                                        |
| [`middleware=`](#middleware)                                                      | Extra middleware appended to the [default stack](#default-stack-main-agent) |
| [`interrupt_on=`](#human-in-the-loop)                                             | Pause before tool calls for human approval                                  |
| [`response_format=`](#structured-output)                                          | Structured output schema                                                    |
| [`state_schema=`](/oss/python/deepagents/context-engineering#custom-state-schema) | Custom graph state schema                                                   |
| [`context_schema=`](/oss/python/deepagents/context-engineering#runtime-context)   | Per-run runtime context schema (user IDs, API keys, feature flags)          |
| [profiles](#profiles)                                                             | Per-model defaults as a reusable bundle                                     |


  ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
  create_deep_agent(
      model: str | BaseChatModel | None = None,
      tools: Sequence[BaseTool | Callable | dict[str, Any]] | None = None,
      *,
      system_prompt: str | SystemMessage | None = None,
      middleware: Sequence[AgentMiddleware] = (),
      subagents: Sequence[SubAgent | CompiledSubAgent | AsyncSubAgent] | None = None,
      skills: list[str] | None = None,
      memory: list[str] | None = None,
      permissions: list[FilesystemPermission] | None = None,
      backend: BackendProtocol | BackendFactory | None = None,
      interrupt_on: dict[str, bool | InterruptOnConfig] | None = None,
      response_format: ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None = None,
      state_schema: type[DeepAgentState] | None = None,
      context_schema: type[ContextT] | None = None,
      checkpointer: Checkpointer | None = None,
      store: BaseStore | None = None,
      debug: bool = False,
      name: str | None = None,
      cache: BaseCache | None = None
  ) -> CompiledStateGraph[AgentState[ResponseT], ContextT, InputAgentState, OutputAgentState[ResponseT]]
  ```




In [4]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
model = ChatNVIDIA(
    model = "meta/llama-3.1-8b-instruct"
)

In [5]:
from tavily import TavilyClient
from typing import Literal
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
def internet_research(
    user_query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a Internet research for a topic."""
    return tavily_client.search(
        query=user_query,
        max_results=max_results,
        topic=topic,
        include_raw_content=include_raw_content,
    )

def mock_weather_tool(city:str) -> str:
    """ Mock Weather tool."""
    return f"Weather in the {city} is always rainy"

In [ ]:
deep_agent = create_deep_agent(
    model=model,
    system_prompt="You are a helpful research assistant",
    tools=[internet_research,mock_weather_tool],
    
    middleware=
)

TypeError: create_deep_agent() got an unexpected keyword argument 'skills'

In [8]:
import os
from typing import Literal

from pydantic import BaseModel, Field
from tavily import TavilyClient

from deepagents import create_deep_agent

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )


class WeatherReport(BaseModel):
    """A structured weather report with current conditions and forecast."""
    location: str = Field(description="The location for this weather report")
    temperature: float = Field(description="Current temperature in Celsius")
    condition: str = Field(
        description="Current weather condition (e.g., sunny, cloudy, rainy)"
    )
    humidity: int = Field(description="Humidity percentage")
    wind_speed: float = Field(description="Wind speed in km/h")
    forecast: str = Field(description="Brief forecast for the next 24 hours")


agent = create_deep_agent(
    model=model,
    response_format=WeatherReport,
    tools=[internet_search],
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the weather like in San Francisco?",
            }
        ]
    }
)

print(result["structured_response"])
# location='San Francisco, California' temperature=18.3 condition='Sunny' humidity=48 wind_speed=7.6 forecast='Pleasant sunny conditions expected to continue with temperatures around 64°F (18°C) during the day, dropping to around 52°F (11°C) at night. Clear skies with minimal precipitation expected.'

location='San Francisco' temperature=15.0 condition='sunny' humidity=60 wind_speed=20.0 forecast='Partly cloudy with a high of 18'
